# Pilot Data Analysis
**Evaluating byT5 parser Performance on 4 repair variant types**

This notebook analyses the PMB-based pilot dataset in which 865 gold sentences wre argumented with four self-repair variants each, parsed by the byT5 parser (Xiao et al., 2025), and evaluated aganist gold SBN using smatch (2013). 

Variant types: 
- **head**: reparandum appers at the start (first ~ 30%) of the sentence
- **mid**: reparandum appears in the middle of the sentence. 
- **tail**: reparandum appears at the end (last ~ 30%). 
- **interrugnum**: self-repair with *I mean* interregnum marker. 

Status codes produced by `evaluate_repair_smatch.py`: 

| Status | Meaning |
|---|---|
| `success` | SBN parsed and smatch computed |
| `ill_formed` | Parser output violates SBN grammar (strict mode) | 
| `parse_error` | Other parsing failure | 
| `gold_error` | Gold SBN itself fails to parse (data quality issue) | 
| `smatch_error` | Parsing succeeded but smatch scoring failed | 

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Define viz style 
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)

VARIANT_COLORS= {
    "head": "#4C72B0",
    "mid": "#55A868",
    "tail": "#C44E52",
    "interrug": "#8172B2"
}

VARIANT_ORDER = ["head", "mid", "tail", "interrug"]
VARIANT_LABELS = {
    "head": "Head", 
    "mid": "Mid", 
    "tail": "Tail", 
    "interrug": "Interrug"
}

In [8]:
# Load data 
DATA_PATH = Path("/Users/hongxuzhou/Documents/GitHub/lct_master_project/evaluated_4_type_repair.tsv")

df = pd.read_csv(DATA_PATH, sep="\t", dtype=str)

# Cast F1 columns to numeric, with empty cells to NaN
F1_COLS = [f"repair_{v}_f1" for v in VARIANT_ORDER]
STATUS_COLS = [f"repair_{v}_status" for v in VARIANT_ORDER]

for col in F1_COLS: 
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Sample length by token of the ORIGINAL sample
df["orig_len"] = df["nl"].str.split().str.len()

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

df[STATUS_COLS + F1_COLS].head(5)

Loaded 865 rows and 20 columns


,repair_head_status,repair_mid_status,repair_tail_status,repair_interrug_status,repair_head_f1,repair_mid_f1,repair_tail_f1,repair_interrug_f1
0,ill_formed,parse_error,parse_error,parse_error,NaN,NaN,NaN,NaN
1,parse_error,success,success,parse_error,NaN,0.754717,0.754717,NaN
2,parse_error,parse_error,parse_error,success,NaN,NaN,NaN,0.366197
3,success,success,success,parse_error,0.90566,0.771930,0.877193,NaN
4,ill_formed,parse_error,parse_error,parse_error,NaN,NaN,NaN,NaN


## 1. Overall Sample Count 


In [9]:
n_sentences = len(df)
n_total = n_sentences * 4 # one record has 4 variant 

print(f"Original sentences: {n_sentences:,}")
print(f"Repair variants: 4 (head / mid / tail / interrug)")
print(f"Total evaluated: {n_total:,}")

Original sentences: 865
Repair variants: 4 (head / mid / tail / interrug)
Total evaluated: 3,460


## 2. Success Rate per Variant


In [15]:
summary_rows = []
for v in VARIANT_ORDER:
    sc = f"repair_{v}_status"
    vc = df[sc].value_counts()
    n_success = vc.get("success", 0)
    summary_rows.append({
        "Variant": VARIANT_LABELS[v], 
        "N": n_sentences, 
        "Success": n_success,
        "Success %": round(100 * n_success / n_sentences, 2),
        **{s: vc.get(s, 0) for s in ["ill_formed", "parse_error", "gold_error", "smatch_error"]},
        })

summary = pd.DataFrame(summary_rows).set_index("Variant")
display(summary)

KeyError: 'tail'